In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00


In [ ]:
import zipfile
import os

def extract_zip(zip_path, extract_to):
    if os.path.exists(zip_path):
        print(f"🗜️ {os.path.basename(zip_path)} 압축 해제 중...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"완료: {extract_to}")
    else:
        print(f"에러: {zip_path} 파일이 /content에 없습니다. 파일명을 확인해 주세요.")

extract_zip('/content/dataset_v6.zip', '/content/')

🗜️ dataset_v6.zip 압축 해제 중...
완료: /content/


In [ ]:
from ultralytics import YOLO

# s 모델로 변경
model = YOLO('yolo26s.pt')

results = model.train(
    data='/content/dataset_v6/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,            # [수정] VRAM 확보를 위해 16으로 조정
    device=0,
    workers=8,
    cache=True,
    pretrained=True,

    # 조기 종료 여유 확보
    patience=25,         # [수정] 조금 더 길게 가져감

    # 파라미터 조정
    cls=2.0,
    box=7.5,
    freeze=0,            # [추가] s 모델은 초기 레이어부터 다시 학습시키는 게 성능상 유리할 수 있음 (또는 5 정도로 낮게)
    dropout=0.2,         # [수정] 일반화 강화
    weight_decay=0.0005,
    optimizer='AdamW',
    lr0=0.0005,          # [수정] 안정적 학습을 위해 하향
    lrf=0.01,
    warmup_epochs=3.0,

    # 데이터 증강 (동일)
    mosaic=1.0,
    mixup=0.15,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,

    iou=0.5,
    close_mosaic=10
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.54 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_v6/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.2, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO
import os

# 1. 학습된 최상의 모델 로드
# 학습이 완료된 경로의 best.pt를 지정하세요
model_path = '/content/runs/detect/train/weights/best.pt'
model = YOLO(model_path)

# 2. 영상 파일 경로
video_path = '/content/Take Time to Take Care (Vehicular Safety).mp4'

# 3. 최적화된 추론 설정
# - conf=0.6: 배경 노이즈(Noisy Box) 억제
# - iou=0.45: 박스 간 겹침 허용 범위 조절
# - agnostic_nms=True: 클래스 간 박스 중첩 허용 (사람/헬멧 동시 검출 핵심)
results = model.predict(
    source=video_path,
    save=True,
    conf=0.6,
    iou=0.45,
    imgsz=640,
    stream=True,
    agnostic_nms=True,
    max_det=100
)

# 4. 결과 처리 및 확인
print("🚀 추론 시작...")
for i, result in enumerate(results):
    # 각 프레임별로 추론이 진행됩니다.
    if i % 30 == 0:  # 30프레임마다 로그 출력
        print(f"처리 중: {i} 프레임 완료")

print("✅ 추론 완료!")
print("결과물은 'runs/detect/predict' 폴더에서 확인하세요.")

🚀 추론 시작...

video 1/1 (frame 1/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 87.4ms
처리 중: 0 프레임 완료
video 1/1 (frame 2/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 12.5ms
video 1/1 (frame 3/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 12.3ms
video 1/1 (frame 4/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 (no detections), 11.4ms
video 1/1 (frame 5/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 11.5ms
video 1/1 (frame 6/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 (no detections), 11.8ms
video 1/1 (frame 7/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 11.6ms
video 1/1 (frame 8/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 1 person, 11.4ms
video 1/1 (frame 9/1434) /content/Take Time to Take Care (Vehicular Safety).mp4: 384x640 (no detections), 11.3m